In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
import joblib

df = pd.read_csv('dataset100k.csv')

# 1. Feature Engineering (El secreto del éxito)
# Ordenamos por máquina y tiempo (asumiendo que el dataset está en orden cronológico por máquina)
df = df.sort_values(by=['id_maquina', 'timestamp_lectura'])

# Calculamos los deltas (cuánto cambió respecto al registro anterior)
df['delta_temp'] = df.groupby('id_maquina')['temp_c'].diff().fillna(0)
df['delta_vibracion'] = df.groupby('id_maquina')['vibracion_mms'].diff().fillna(0)

# 2. Selección de variables
features = ['rpm', 'vibracion_mms', 'temp_c', 'corriente_motor_a', 'delta_temp', 'delta_vibracion']
X = df[features]
y = df['etiqueta_prediccion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Entrenamiento priorizando Recall
# Usamos un peso personalizado: 1 para Normal, 3 para Riesgo (penalizamos más perder una falla)
pesos_clases = {0: 1, 1: 3}
rf_model = RandomForestClassifier(n_estimators=100, class_weight=pesos_clases, random_state=42, max_depth=12)
rf_model.fit(X_train, y_train)

# 4. Predicciones con Umbral Ajustado (0.40)
y_prob = rf_model.predict_proba(X_test)[:, 1]
umbral = 0.40
y_pred_ajustado = (y_prob >= umbral).astype(int)

# 5. Evaluación
print("\n--- 📊 MÉTRICAS V2.0 (Con Deltas y Umbral 0.40) ---")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_test, y_pred_ajustado))
print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred_ajustado))

joblib.dump(rf_model, "modelo_v2_cnc.pkl")


--- 📊 MÉTRICAS V2.0 (Con Deltas y Umbral 0.40) ---
ROC AUC Score: 0.9558

Reporte de Clasificación Detallado:
              precision    recall  f1-score   support

           0       0.98      0.86      0.92     23833
           1       0.63      0.94      0.76      6167

    accuracy                           0.88     30000
   macro avg       0.81      0.90      0.84     30000
weighted avg       0.91      0.88      0.88     30000

Matriz de Confusión:
 [[20484  3349]
 [  357  5810]]


['modelo_v2_cnc.pkl']